# 02 - Limpieza y preparación

## Importar librerías

In [1]:
from pathlib import Path
from zipfile import ZipFile
from html.parser import HTMLParser
from IPython.display import display
import xml.etree.ElementTree as ET
import unicodedata
import html
import json
import re
import numpy as np
import pandas as pd

## Mejorar la visualización

In [2]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## Establecer las rutas

In [3]:
RAIZ = Path.cwd().resolve()
for carpeta in [RAIZ, *RAIZ.parents]:
    if (carpeta / 'requirements.txt').is_file() and (carpeta / 'data').is_dir():
        RAIZ = carpeta
        break
else:
    raise FileNotFoundError('Abra el notebook desde el proyecto.')

RAW = RAIZ / 'data/raw'
PROCESSED = RAIZ / 'data/processed'
REPORTS = RAIZ / 'data/reports/02_limpieza_preparacion'
ANIOS = list(range(2017, 2025))
VERSION = '202601'
claves = ['iso3_o', 'iso3_d', 'year']

## Seleccionar las exportaciones de Colombia

In [4]:
bases = []
with ZipFile(RAW / 'baci' / f'BACI_HS17_V{VERSION}.zip') as paquete:
    for anio in ANIOS:
        partes = []
        with paquete.open(f'BACI_HS17_Y{anio}_V{VERSION}.csv') as entrada:
            for bloque in pd.read_csv(entrada, dtype = {'k': 'string'}, float_precision = 'round_trip', chunksize = 500_000):
                partes.append(bloque.loc[bloque['i'].eq(170)].copy())
        parte = pd.concat(partes, ignore_index = True)
        parte[['t', 'i', 'j']] = parte[['t', 'i', 'j']].astype('int16')
        bases.append(parte)
baci = pd.concat(bases, ignore_index = True)
baci.rename(columns = {'t': 'year', 'i': 'reporter_code', 'j': 'partner_code',
    'k': 'hs6', 'v': 'trade_value_thousand_usd', 'q': 'quantity_tonnes'}, inplace = True)
baci['hs6'] = baci['hs6'].astype('string').str.strip().str.zfill(6)
baci['hs4'] = baci['hs6'].str[:4]
baci['hs2'] = baci['hs6'].str[:2]
baci['trade_value_usd'] = baci['trade_value_thousand_usd'] * 1_000
baci['quantity_kg'] = baci['quantity_tonnes'] * 1_000
baci = baci.assign(flow = 'X', frequency = 'A', hs_revision = 'HS2017', source = 'BACI', source_version = VERSION)
baci.head()

,year,reporter_code,partner_code,hs6,trade_value_thousand_usd,quantity_tonnes,hs4,hs2,trade_value_usd,quantity_kg,flow,frequency,hs_revision,source,source_version
0,2017,170,8,080390,"1,676.91","1,933.51",0803,08,"1,676,907.00","1,933,513.00",X,A,HS2017,BACI,202601
1,2017,170,8,090111,145.49,40.21,0901,09,"145,493.00","40,210.00",X,A,HS2017,BACI,202601
2,2017,170,8,170114,15.52,25.00,1701,17,"15,523.00","25,000.00",X,A,HS2017,BACI,202601
3,2017,170,8,250700,50.00,100.00,2507,25,"50,000.00","100,000.00",X,A,HS2017,BACI,202601
4,2017,170,8,270400,946.76,"5,917.23",2704,27,"946,757.00","5,917,230.00",X,A,HS2017,BACI,202601


## Incorporar los catálogos BACI

In [5]:
paises = pd.read_csv(RAW / 'baci' / f'country_codes_V{VERSION}.csv', dtype = {'country_iso3': 'string'})
paises['country_iso3'] = paises['country_iso3'].str.strip().str.upper()
productos = pd.read_csv(RAW / 'baci' / f'product_codes_HS17_V{VERSION}.csv', dtype = {'code': 'string'})
productos['code'] = productos['code'].str.strip().str.zfill(6)
baci = baci.merge(paises[['country_code', 'country_iso3']].rename(columns = {
    'country_code': 'reporter_code', 'country_iso3': 'reporter_iso3'}),
    on = 'reporter_code', how = 'left', validate = 'many_to_one')
baci = baci.merge(paises[['country_code', 'country_iso3', 'country_name']].rename(columns = {
    'country_code': 'partner_code', 'country_iso3': 'partner_iso3', 'country_name': 'partner_name'}),
    on = 'partner_code', how = 'left', validate = 'many_to_one')
baci = baci.merge(productos[['code', 'description']].rename(columns = {'code': 'hs6', 'description': 'hs6_name'}),
    on = 'hs6', how = 'left', validate = 'many_to_one')
baci = baci[['year', 'reporter_code', 'reporter_iso3', 'partner_code', 'partner_iso3', 'partner_name',
    'flow', 'frequency', 'hs6', 'hs6_name', 'hs4', 'hs2', 'hs_revision', 'trade_value_thousand_usd',
    'trade_value_usd', 'quantity_tonnes', 'quantity_kg', 'source', 'source_version']]
baci = baci.sort_values(['year', 'partner_code', 'hs6']).reset_index(drop = True)
assert not baci.duplicated(['year', 'reporter_code', 'partner_code', 'hs6']).any()
baci.shape

(412454, 19)

## Leer el catálogo de territorios

In [6]:
class _TablaM49(HTMLParser):
    def __init__(self):
        super().__init__()
        self.dentro = False
        self.celda = False
        self.filas = []
        self.fila = []
        self.textos = []

    def handle_starttag(self, tag, attrs):
        if tag == "table" and dict(attrs).get("id") == "downloadTableES":
            self.dentro = True
        if self.dentro and tag == "tr":
            self.fila = []
        if self.dentro and tag in ["td", "th"]:
            self.celda, self.textos = True, []

    def handle_data(self, data):
        if self.dentro and self.celda:
            self.textos.append(data)

    def handle_endtag(self, tag):
        if self.dentro and tag in ["td", "th"]:
            self.fila.append(" ".join(" ".join(self.textos).split()))
            self.celda = False
        if self.dentro and tag == "tr":
            self.filas.append(self.fila)
        if tag == "table":
            self.dentro = False

parser = _TablaM49()
parser.feed((RAW / 'variables/complementos_finales/un_m49.html').read_text(encoding='utf-8'))
m49 = pd.DataFrame(parser.filas[1:], columns = parser.filas[0]).rename(columns = {
    'ISO-alpha3 Code': 'iso3_referencia', 'M49 Code': 'codigo_m49', 'Country or Area': 'nombre_m49',
    'Region Name': 'region_m49', 'Sub-region Name': 'subregion_m49'})
m49 = m49[['iso3_referencia', 'codigo_m49', 'nombre_m49', 'region_m49', 'subregion_m49']].replace('', pd.NA)
assert m49['iso3_referencia'].is_unique and m49['iso3_referencia'].notna().all()

## Homologar los destinos

In [7]:
destinos = baci[['partner_code', 'partner_iso3']].drop_duplicates().merge(
    paises, left_on = 'partner_code', right_on = 'country_code', validate = 'one_to_one')
destinos.rename(columns = {'partner_code': 'codigo_baci', 'partner_iso3': 'iso3_d',
    'country_name': 'nombre_baci_original'}, inplace = True)
destinos['iso3_referencia'] = destinos['iso3_d'].astype(str).replace({'S19': 'TWN'})
destinos['tipo_equivalencia'] = np.where(destinos['iso3_d'].eq('S19'), 'proxy_estadistico_autorizado', 'coincidencia_codigo')
destinos = destinos.merge(m49, on = 'iso3_referencia', how = 'left', validate = 'one_to_one')
destinos['nombre_destino'] = destinos['nombre_m49']
destinos.loc[destinos['iso3_d'].eq('S19'), 'nombre_destino'] = 'Taiwán'
destinos['nombre_destino'] = destinos['nombre_destino'].map(lambda x: unicodedata.normalize('NFC', html.unescape(x)))
destinos = destinos[['iso3_d', 'iso3_referencia', 'codigo_baci', 'codigo_m49', 'nombre_destino',
    'nombre_m49', 'nombre_baci_original', 'tipo_equivalencia', 'region_m49', 'subregion_m49']]
destinos = destinos.sort_values('iso3_d').reset_index(drop = True)
assert destinos['iso3_d'].is_unique and destinos['nombre_destino'].notna().all()
destinos.head()

,iso3_d,iso3_referencia,codigo_baci,codigo_m49,nombre_destino,nombre_m49,nombre_baci_original,tipo_equivalencia,region_m49,subregion_m49
0,ABW,ABW,533,533,Aruba,Aruba,Aruba,coincidencia_codigo,Américas,América Latina y el Caribe
1,AFG,AFG,4,004,Afganistán,Afganistán,Afghanistan,coincidencia_codigo,Asia,Asia meridional
2,AGO,AGO,24,024,Angola,Angola,Angola,coincidencia_codigo,África,África Subsahariana
3,AIA,AIA,660,660,Anguila,Anguila,Anguilla,coincidencia_codigo,Américas,América Latina y el Caribe
4,ALB,ALB,8,008,Albania,Albania,Albania,coincidencia_codigo,Europa,Europa meridional


## Preparar las variables WDI

In [8]:
carpeta_wdi = RAW / 'variables/world_bank'
registros = json.loads((carpeta_wdi / 'countries.json').read_text(encoding='utf-8'))['registros']
paises_wdi = pd.DataFrame([{'iso3_d': r['id'], 'country_name_wdi': r['name'], 'region_wdi': r['region']['value']}
    for r in registros if r['region']['value'] != 'Aggregates'])
wdi = paises_wdi.merge(pd.DataFrame({'year': ANIOS}), how = 'cross')
indicadores = {'NY.GDP.MKTP.KD': 'gdp_real_usd'}
for codigo, variable in indicadores.items():
    registros = json.loads((carpeta_wdi / f'{codigo}_2017_2024.json').read_text(encoding='utf-8'))['registros']
    serie = pd.DataFrame([{'iso3_d': r['countryiso3code'], 'year': int(r['date']), variable: r['value']} for r in registros])
    serie = serie.loc[serie['iso3_d'].isin(paises_wdi['iso3_d'])]
    serie[variable] = pd.to_numeric(serie[variable], errors = 'raise')
    wdi = wdi.merge(serie, on = ['iso3_d', 'year'], how = 'left', validate = 'one_to_one')
wdi = wdi.sort_values(['year', 'iso3_d']).reset_index(drop = True)
wdi.head()

,iso3_d,country_name_wdi,region_wdi,year,gdp_real_usd
0,ABW,Aruba,Latin America & Caribbean,2017,"3,104,263,601.10"
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",2017,"20,084,646,751.50"
2,AGO,Angola,Sub-Saharan Africa,2017,"102,135,580,392.60"
3,ALB,Albania,Europe & Central Asia,2017,"12,309,839,868.71"
4,AND,Andorra,Europe & Central Asia,2017,"2,903,390,035.84"


## Preparar la geografía de Colombia

In [9]:
columnas_geo = ['year', 'country_id_o', 'country_id_d', 'iso3_o', 'iso3_d', 'country_exists_o', 'country_exists_d',
    'dist', 'distcap', 'distw_harmonic', 'distw_arithmetic', 'contig', 'comlang_off', 'comlang_ethno']
partes = []
with ZipFile(RAW / 'variables/oficiales/Gravity_csv_V202211.zip') as paquete:
    with paquete.open('Gravity_V202211.csv') as entrada:
        for bloque in pd.read_csv(entrada, usecols = columnas_geo, chunksize = 250_000):
            vigente = (bloque['year'].isin(ANIOS) & bloque['iso3_o'].eq('COL') & bloque['iso3_d'].ne('COL')
                & bloque['country_exists_o'].eq(1) & bloque['country_exists_d'].eq(1))
            partes.append(bloque.loc[vigente].copy())
geografia = pd.concat(partes, ignore_index = True).drop(columns = ['country_exists_o', 'country_exists_d'])
ultimo_anio = int(geografia['year'].max())
ultimas = geografia.sort_values('year').drop_duplicates('iso3_d', keep = 'last')
geografia['geography_reference_year'] = geografia['year']
geografia['geography_static_extension'] = False
for anio in ANIOS:
    if anio > ultimo_anio:
        extension = ultimas.copy()
        extension['geography_reference_year'] = extension['year']
        extension['year'] = anio
        extension['geography_static_extension'] = True
        geografia = pd.concat([geografia, extension], ignore_index = True)
geografia['geography_source'] = 'CEPII Gravity V202211'
assert not geografia.duplicated(['iso3_d', 'year']).any()

## Definir las equivalencias de membresía

In [10]:
equivalencias_omc = {'CHT': 'TWN', 'EEC': 'EUU', 'ROM': 'ROU'}

equivalencias_ocde = {
    'Australia': 'AUS',
    'Austria': 'AUT',
    'Belgium': 'BEL',
    'Canada': 'CAN',
    'Chile': 'CHL',
    'Colombia': 'COL',
    'Costa Rica': 'CRI',
    'Czech Republic': 'CZE',
    'Denmark': 'DNK',
    'Estonia': 'EST',
    'Finland': 'FIN',
    'France': 'FRA',
    'Germany': 'DEU',
    'Greece': 'GRC',
    'Hungary': 'HUN',
    'Iceland': 'ISL',
    'Ireland': 'IRL',
    'Israel': 'ISR',
    'Italy': 'ITA',
    'Japan': 'JPN',
    'Korea': 'KOR',
    'Latvia': 'LVA',
    'Lithuania': 'LTU',
    'Luxembourg': 'LUX',
    'Mexico': 'MEX',
    'Netherlands': 'NLD',
    'New Zealand': 'NZL',
    'Norway': 'NOR',
    'Poland': 'POL',
    'Portugal': 'PRT',
    'Slovak Republic': 'SVK',
    'Slovenia': 'SVN',
    'Spain': 'ESP',
    'Sweden': 'SWE',
    'Switzerland': 'CHE',
    'Türkiye': 'TUR',
    'United Kingdom': 'GBR',
    'United States': 'USA',
}

## Preparar las membresías

In [11]:
texto = (RAW / 'variables/oficiales/wto_news_vars.js').read_text(encoding='cp1252')
bloques = re.findall(r'country\[country.length\]\s*=\s*\{(.*?)\};', texto, re.S)
campos = [dict(re.findall(r'(\w+)\s*:\s*"((?:[^"\\]|\\.)*)"', b)) for b in bloques]
omc = pd.DataFrame(campos)[['code', 'name', 'status', 'joindate']].rename(columns = {
    'code': 'wto_code', 'name': 'wto_name', 'status': 'wto_status'})
omc['iso3_d'] = omc['wto_code'].replace(equivalencias_omc)
omc['wto_accession_date'] = pd.to_datetime(omc['joindate'].str[:10].str.replace('.', '-', regex = False), errors = 'coerce')
omc.drop(columns = 'joindate', inplace = True)
registros = json.loads((RAW / 'variables/oficiales/oecd_lists.json').read_text(encoding='utf-8'))['countries']
ocde = pd.DataFrame([r for r in registros if r.get('kind') == 'Member'])
ocde['iso3_d'] = ocde['name'].map(equivalencias_ocde)
ocde['oecd_accession_date'] = pd.to_datetime(ocde['accessionDate'], errors = 'raise')
ocde = ocde[['iso3_d', 'name', 'oecd_accession_date']].rename(columns = {'name': 'oecd_name'})
assert not omc['iso3_d'].duplicated().any() and ocde['iso3_d'].notna().all()

## Definir los acuerdos comerciales

In [12]:
acuerdos = [
    ('CAN', ['BOL', 'ECU', 'PER'], '2017-01-01', None, 'customs_union'),
    ('MERCOSUR_ACE59', ['ARG', 'BRA', 'URY'], '2005-02-01', None, 'free_trade_area'),
    ('MERCOSUR_ACE59', ['PRY'], '2005-04-19', None, 'free_trade_area'),
    ('MEXICO', ['MEX'], '1995-01-01', None, 'free_trade_area'),
    ('CHILE', ['CHL'], '2009-05-08', None, 'free_trade_area'),
    ('TRIANGULO_NORTE', ['GTM'], '2009-11-13', None, 'free_trade_area'),
    ('TRIANGULO_NORTE', ['SLV'], '2010-02-01', None, 'free_trade_area'),
    ('TRIANGULO_NORTE', ['HND'], '2010-03-27', None, 'free_trade_area'),
    ('EFTA', ['CHE', 'LIE'], '2011-07-01', None, 'free_trade_area'),
    ('EFTA', ['NOR'], '2014-09-01', None, 'free_trade_area'),
    ('EFTA', ['ISL'], '2014-10-01', None, 'free_trade_area'),
    ('CANADA', ['CAN'], '2011-08-15', None, 'free_trade_area'),
    ('ESTADOS_UNIDOS', ['USA'], '2012-05-15', None, 'free_trade_area'),
    ('UE', ['AUT', 'BEL', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HUN', 'IRL', 'ITA', 'LTU', 'LUX', 'LVA', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE'], '2013-08-01', None, 'provisional_application'),
    ('UE_CROACIA_EXPORTACIONES_COL', ['HRV'], '2013-08-01', None, 'export_preference'),
    ('UE_REINO_UNIDO', ['GBR'], '2013-08-01', '2020-12-31', 'provisional_application'),
    ('REINO_UNIDO_CONTINUIDAD', ['GBR'], '2021-01-01', '2022-06-27', 'bridge'),
    ('REINO_UNIDO', ['GBR'], '2022-06-28', None, 'free_trade_area'),
    ('COREA', ['KOR'], '2016-07-15', None, 'free_trade_area'),
    ('COSTA_RICA', ['CRI'], '2016-08-01', None, 'free_trade_area'),
    ('ISRAEL', ['ISR'], '2020-08-11', None, 'free_trade_area'),
]
eventos = pd.DataFrame(acuerdos, columns = ['agreement_id', 'iso3_d', 'start_date', 'end_date', 'kind']).explode('iso3_d')
eventos[['start_date', 'end_date']] = eventos[['start_date', 'end_date']].apply(pd.to_datetime)

## Construir las variables por destino y año

In [13]:
base = destinos[['iso3_d', 'iso3_referencia']].merge(pd.DataFrame({'year': ANIOS}), how = 'cross')
base['iso3_d'] = base['iso3_d'].astype(str)
base['iso3_o'] = 'COL'
base = base.merge(wdi, on = ['iso3_d', 'year'], how = 'left', validate = 'one_to_one')
base = base.merge(geografia.rename(columns = {'iso3_d': 'iso3_referencia'}),
    on = ['iso3_o', 'iso3_referencia', 'year'], how = 'left', validate = 'one_to_one')
base = base.merge(omc, on = 'iso3_d', how = 'left', validate = 'many_to_one')
base = base.merge(ocde, on = 'iso3_d', how = 'left', validate = 'many_to_one')
cierre = pd.to_datetime(base['year'].astype(str) + '-12-31')
inicio = pd.to_datetime(base['year'].astype(str) + '-01-01')
for prefijo in ['wto', 'oecd']:
    fecha = base[prefijo + '_accession_date']
    base[prefijo + '_d'] = (fecha.notna() & fecha.le(cierre)).astype('Int8')
    base[prefijo + '_full_year'] = (fecha.notna() & fecha.le(inicio)).astype('Int8')
    base[prefijo + '_exposure'] = [max(0, (fin - max(f, ini)).days + 1) / ((fin - ini).days + 1)
        if pd.notna(f) and f <= fin else 0.0 for f, ini, fin in zip(fecha, inicio, cierre)]

politicas = []
for iso3 in destinos['iso3_d']:
    grupo = eventos.loc[eventos['iso3_d'].eq(iso3)]
    for anio in ANIOS:
        dias = pd.date_range(f'{anio}-01-01', f'{anio}-12-31')
        cobertura, formales, ids = np.zeros(len(dias), dtype=bool), np.zeros(len(dias), dtype=bool), []
        for evento in grupo.itertuples(index = False):
            vigente = (dias >= evento.start_date) & (dias <= evento.end_date if pd.notna(evento.end_date) else True)
            cobertura |= vigente
            if evento.kind not in ['bridge', 'export_preference']:
                formales |= vigente
            if vigente.any():
                ids.append(evento.agreement_id)
        politicas.append([iso3, anio, int(cobertura[-1]), int(cobertura.all()), cobertura.mean(),
            int(formales[-1]), formales.mean(), ';'.join(sorted(set(ids)))])
columnas_fta = ['fta_in_force', 'fta_full_year', 'fta_exposure', 'fta_formal_in_force', 'fta_formal_exposure']
base = base.merge(pd.DataFrame(politicas, columns = ['iso3_d', 'year', *columnas_fta, 'fta_agreements']),
    on = ['iso3_d', 'year'], validate = 'one_to_one')

## Aplicar las correspondencias territoriales

In [14]:
sin_cobertura_fta = ['ABW', 'AIA', 'ASM', 'BES', 'BMU', 'CCK', 'CUW', 'CXR', 'CYM', 'FLK', 'FRO', 'GIB', 'GRL', 'GUM', 'IOT',
 'MNP', 'MSR', 'MYT', 'NCL', 'NFK', 'NIU', 'PCN', 'PRI', 'PYF', 'SHN', 'SMR', 'SPM', 'TCA', 'TKL', 'VGB',
 'VIR', 'WLF']
base.loc[base['iso3_d'].isin(sin_cobertura_fta), columnas_fta] = np.nan
base['fta_scope_verified'] = ~base['iso3_d'].isin(sin_cobertura_fta) & base['iso3_d'].ne('S19')
resueltos = ['ABW', 'BES', 'CUW', 'GRL', 'NCL', 'PYF', 'AIA', 'BMU', 'CYM', 'IOT', 'MSR', 'TCA', 'VGB', 'ASM', 'GUM', 'MNP', 'S19']
for columna in columnas_fta:
    mascara = base['iso3_d'].isin(resueltos) & base[columna].isna()
    base.loc[mascara, columna] = 0
base.loc[base['iso3_d'].isin(resueltos), 'fta_scope_verified'] = True
base.loc[base['iso3_d'].eq('HRV'), ['fta_formal_in_force', 'fta_formal_exposure']] = np.nan
base.loc[base['iso3_d'].isin(['BES', 'CUW']) & base['comlang_off'].isna(), 'comlang_off'] = 0
miembros = omc.loc[omc['wto_status'].eq('MBR')].set_index('iso3_d')
ingreso = base['iso3_referencia'].map(miembros['wto_accession_date'])
base['wto_d'] = (ingreso.notna() & ingreso.le(cierre)).astype('Int8')
es_taiwan = base['iso3_d'].eq('S19')
base.loc[es_taiwan, 'country_id_o'] = np.nan
base.loc[es_taiwan, 'wto_accession_date'] = ingreso.loc[es_taiwan]
base.loc[es_taiwan, ['wto_code', 'wto_name', 'wto_status']] = ['CHT', 'Chinese Taipei', 'MBR']
base.loc[es_taiwan, ['wto_full_year', 'wto_exposure']] = 1
base.loc[es_taiwan, ['oecd_d', 'oecd_full_year', 'oecd_exposure']] = 0
base.loc[base['wto_d'].eq(0), ['wto_full_year', 'wto_exposure']] = 0

## Preparar el complemento de Taiwán

In [15]:
registros = []
for obs in ET.parse(RAW / 'variables/complementos_finales/taiwan_cuentas_nacionales.xml').getroot().findall('Obs'):
    anio = obs.findtext('TIME_PERIOD', '')
    if obs.findtext('FREQ') == 'Y' and obs.findtext('TYPE') == '原始值' and anio.isdigit() and 2015 <= int(anio) <= 2024 and obs.findtext('Item') in ['國內生產毛額GDP(名目值，百萬美元)', '經濟成長率(%)']:
        registros.append({'year': int(anio), 'item': obs.findtext('Item'), 'value': obs.findtext('Item_VALUE')})
tabla = pd.DataFrame(registros)
tabla['value'] = pd.to_numeric(tabla['value'], errors = 'raise')
tabla = tabla.pivot(index = 'year', columns = 'item', values = 'value').sort_index()
nominal = tabla['國內生產毛額GDP(名目值，百萬美元)'] * 1_000_000
real = {2015: float(nominal.loc[2015])}
for anio, tasa in tabla.loc[2016:2024, '經濟成長率(%)'].items():
    real[anio] = real[anio - 1] * (1 + tasa / 100)
taiwan = pd.DataFrame({'year': ANIOS, 'iso3_d': 'S19'})
taiwan['gdp_real_usd'] = taiwan['year'].map(real)

## Preparar el complemento de BES

In [16]:
carpeta = RAW / 'variables/complementos_finales'
temas = pd.DataFrame(json.loads((carpeta / 'cbs_topics.json').read_text(encoding='utf-8'))['value']).set_index('Key')
territorios = pd.DataFrame(json.loads((carpeta / 'cbs_territorios.json').read_text(encoding='utf-8'))['value'])
assert territorios.assign(Key = territorios['Key'].str.strip()).set_index('Key').loc['CN01', 'Title'] == 'Caribbean Netherlands'
assert temas.loc[['GDPValueAtCurrentPrices_1', 'GDPValueAtPricesOf2017_2'], 'Unit'].eq('million USD').all()
tabla = pd.DataFrame(json.loads((carpeta / 'cbs_data.json').read_text(encoding='utf-8'))['value'])
tabla = tabla.loc[tabla['CarribeanNetherlands'].str.strip().eq('CN01')].copy()
tabla['year'] = tabla['Periods'].str[:4].astype(int)
tabla = tabla.set_index('year').sort_index()
factor = tabla.loc[2015, 'GDPValueAtCurrentPrices_1'] / tabla.loc[2015, 'GDPValueAtPricesOf2017_2']
bes = pd.DataFrame({'year': ANIOS, 'iso3_d': 'BES'})
bes['gdp_real_usd'] = bes['year'].map(tabla['GDPValueAtPricesOf2017_2']) * factor * 1_000_000
suplementos = pd.concat([taiwan, bes], ignore_index = True)
serie = suplementos.set_index(['iso3_d', 'year'])['gdp_real_usd']
valores = serie.reindex(pd.MultiIndex.from_frame(base[['iso3_d', 'year']])).to_numpy()
base['gdp_real_usd'] = base['gdp_real_usd'].fillna(pd.Series(valores, index = base.index))

## Organizar las covariables

In [17]:
base = base.merge(destinos[['iso3_d', 'nombre_destino']], on = 'iso3_d', validate = 'many_to_one')
base['year'] = base['year'].astype('int16')
base['iso3_d'] = base['iso3_d'].astype('str')
variables = ['gdp_real_usd', 'dist', 'contig', 'comlang_off', 'fta_in_force', 'wto_d', 'oecd_d']
for columna in ['contig', 'comlang_off', 'fta_in_force', 'wto_d', 'oecd_d', 'fta_full_year', 'fta_formal_in_force']:
    base[columna] = base[columna].astype('Int8')
base = base.sort_values(['year', 'iso3_d']).reset_index(drop = True)
base['estado_datos'] = np.where(base[variables].notna().all(axis=1), 'completo', 'pendiente')
final = base[[*claves[:2], 'nombre_destino', 'year', *variables, 'estado_datos']].copy()
final.head()

,iso3_o,iso3_d,nombre_destino,year,gdp_real_usd,dist,contig,comlang_off,fta_in_force,wto_d,oecd_d,estado_datos
0,COL,ABW,Aruba,2017,"3,104,263,601.10",982.00,0,1,0,0,0,completo
1,COL,AFG,Afganistán,2017,"20,084,646,751.50","14,220.00",0,0,0,1,0,completo
2,COL,AGO,Angola,2017,"102,135,580,392.60","9,802.00",0,0,0,1,0,completo
3,COL,AIA,Anguila,2017,NaN,"1,925.00",0,0,0,0,0,pendiente
4,COL,ALB,Albania,2017,"12,309,839,868.71","10,001.00",0,0,0,1,0,completo


## Identificar los años con información faltante

In [18]:
# Años con información faltante por destino y variable
# Se usa la tabla en memoria para evitar leer la salida de otra ejecución.
base_revision = final.copy()

variables = [
    "gdp_real_usd", "dist", "contig", "comlang_off",
    "fta_in_force", "wto_d", "oecd_d"
]

revision = base_revision.melt(
    id_vars=["iso3_d", "nombre_destino", "year"],
    value_vars=variables,
    var_name="variable",
    value_name="valor"
)

faltantes_por_anio = (
    revision.loc[revision["valor"].isna()]
    .groupby(["iso3_d", "nombre_destino", "variable"], as_index=False)
    .agg(
        anios_faltantes=("year", lambda x: ", ".join(
            str(anio) for anio in sorted(x.unique())
        )),
        cantidad_anios=("year", "nunique")
    )
    .sort_values(["iso3_d", "variable"])
    .reset_index(drop=True)
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(faltantes_por_anio)

,iso3_d,nombre_destino,variable,anios_faltantes,cantidad_anios
0,AIA,Anguila,gdp_real_usd,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",8
1,ASM,Samoa Americana,comlang_off,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",8
2,ASM,Samoa Americana,gdp_real_usd,"2023, 2024",2
3,GIB,Gibraltar,fta_in_force,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",8
4,GIB,Gibraltar,gdp_real_usd,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",8
5,GRL,Groenlandia,gdp_real_usd,2024,1
6,GUM,Guam,comlang_off,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",8
7,GUM,Guam,gdp_real_usd,"2023, 2024",2
8,IOT,Territorio Británico del Océano Índico,comlang_off,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",8
9,IOT,Territorio Británico del Océano Índico,gdp_real_usd,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",8


## Excluir destinos sin PIB en todo el periodo

In [19]:
final_sin_imputar = final.copy()
sin_pib = final.groupby('iso3_d')['gdp_real_usd'].count().eq(0)
codigos_excluidos = sin_pib.index[sin_pib].tolist()
destinos_excluidos = destinos.loc[destinos['iso3_d'].isin(codigos_excluidos)].copy()
final = final.loc[~final['iso3_d'].isin(codigos_excluidos)].copy().reset_index(drop = True)
base = base.loc[~base['iso3_d'].isin(codigos_excluidos)].copy().reset_index(drop = True)
destinos = destinos.loc[~destinos['iso3_d'].isin(codigos_excluidos)].reset_index(drop = True)
destinos_excluidos[['iso3_d', 'nombre_destino']]

,iso3_d,nombre_destino
3,AIA,Anguila
68,GIB,Gibraltar
86,IOT,Territorio Británico del Océano Índico
129,MSR,Montserrat
134,NCL,Nueva Caledonia
149,PRK,República Popular Democrática de Corea
196,VGB,Islas Vírgenes Británicas


## Calcular el crecimiento de la muestra mundial

In [20]:
serie_wdi = wdi.set_index(['iso3_d', 'year'])['gdp_real_usd']
serie_complementos = suplementos.set_index(['iso3_d', 'year'])['gdp_real_usd']
panel_pib = serie_wdi.combine_first(serie_complementos).unstack('year').reindex(columns = ANIOS)
panel_pib = panel_pib.loc[panel_pib.notna().all(axis=1) & panel_pib.gt(0).all(axis=1) & np.isfinite(panel_pib).all(axis=1)]
tasas_mundiales = panel_pib.sum(axis=0).pct_change(fill_method = None)
assert len(panel_pib) == 201 and tasas_mundiales.loc[2018:2024].notna().all()

## Completar los faltantes acordados

In [21]:
imputaciones = []
for iso3, grupo in final.groupby('iso3_d'):
    grupo = grupo.sort_values('year')
    conocidos = grupo.loc[grupo['gdp_real_usd'].notna()]
    ultimo_anio = int(conocidos['year'].max())
    ultimo_valor = float(conocidos.iloc[-1]['gdp_real_usd'])
    pendientes = grupo.loc[grupo['gdp_real_usd'].isna()]
    assert pendientes['year'].gt(ultimo_anio).all(), f'Hueco inicial o interno: {iso3}'
    for indice, fila in pendientes.iterrows():
        anio = int(fila['year'])
        assert anio == ultimo_anio + 1
        ultimo_valor *= 1 + float(tasas_mundiales.loc[anio])
        final.loc[indice, 'gdp_real_usd'] = ultimo_valor
        imputaciones.append([iso3, anio, 'gdp_real_usd', ultimo_valor])
        ultimo_anio = anio
for columna in ['comlang_off', 'fta_in_force']:
    for fila in final.loc[final[columna].isna(), ['iso3_d', 'year']].itertuples(index=False):
        imputaciones.append([fila.iso3_d, fila.year, columna, 0.0])
    final[columna] = final[columna].fillna(0).astype('Int8')
imputaciones = pd.DataFrame(imputaciones, columns = ['iso3_d', 'year', 'variable', 'valor_final'])
for columna in ['gdp_real_usd', 'comlang_off', 'fta_in_force']:
    indices = pd.MultiIndex.from_frame(imputaciones.loc[imputaciones['variable'].eq(columna), ['iso3_d', 'year']])
    base[columna + '_imputado'] = pd.MultiIndex.from_frame(final[['iso3_d', 'year']]).isin(indices)
base['alguna_imputacion_arbitraria'] = base.filter(like='_imputado').any(axis=1)
final['estado_datos'] = np.where(base['alguna_imputacion_arbitraria'], 'completo_con_imputacion', 'completo')
assert final.shape == (1576, 12) and final[variables].notna().all().all()
assert len(imputaciones) == 40 and len(destinos_excluidos) == 7
final.head()

,iso3_o,iso3_d,nombre_destino,year,gdp_real_usd,dist,contig,comlang_off,fta_in_force,wto_d,oecd_d,estado_datos
0,COL,ABW,Aruba,2017,"3,104,263,601.10",982.00,0,1,0,0,0,completo
1,COL,AFG,Afganistán,2017,"20,084,646,751.50","14,220.00",0,0,0,1,0,completo
2,COL,AGO,Angola,2017,"102,135,580,392.60","9,802.00",0,0,0,1,0,completo
3,COL,ALB,Albania,2017,"12,309,839,868.71","10,001.00",0,0,0,1,0,completo
4,COL,AND,Andorra,2017,"2,903,390,035.84","8,486.00",0,1,0,0,0,completo


## Comprobar y guardar las bases

In [22]:
salidas = {
    'baci/baci_exports_COL_2017_2024_HS2017_V202601.parquet': baci,
    'variables/base_final_variables_COL_2017_2024.parquet': final,
    'variables/catalogo_destinos_variables.parquet': destinos,
}
for nombre, tabla in salidas.items():
    archivo = PROCESSED / nombre
    if archivo.exists():
        pd.testing.assert_frame_equal(tabla, pd.read_parquet(archivo), check_exact = False, check_dtype = False, rtol = 1e-12, atol = 1e-6)
    else:
        archivo.parent.mkdir(parents=True, exist_ok=True)
        tabla.to_parquet(archivo, index = False, compression = 'zstd')

REPORTS.mkdir(parents=True, exist_ok=True)
for nombre, tabla in {'faltantes_antes': faltantes_por_anio, 'destinos_excluidos': destinos_excluidos,
                      'imputaciones': imputaciones, 'base_sin_imputar': final_sin_imputar}.items():
    tabla.to_parquet(REPORTS / f'{nombre}.parquet', index = False, compression = 'zstd')
pd.DataFrame([{'base': nombre, 'filas': len(tabla), 'columnas': len(tabla.columns)} for nombre, tabla in salidas.items()])

,base,filas,columnas
0,baci/baci_exports_COL_2017_2024_HS2017_V202601...,412454,19
1,variables/base_final_variables_COL_2017_2024.p...,1576,12
2,variables/catalogo_destinos_variables.parquet,197,10
